# Multi-Entity Biomedical NER — Experiment Runner

One-click notebook to launch every training configuration (1..6) with three
seeds, then collate the per-seed evaluation JSONs into mean ± std tables.

Each section is independent — feel free to skip cells you do not need.
Designed for Vast.ai (T4 16GB or 3080 Ti 12GB) but the launch commands are
identical on any Linux host.

## 1. Environment check

In [ ]:
!nvidia-smi || echo "No GPU detected. CPU runs will be very slow."
import sys; print("Python:", sys.version)
import torch; print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
import transformers; print("Transformers:", transformers.__version__)

## 2. Install requirements (run once per fresh machine)

In [ ]:
!pip install -r requirements.txt

## 3. Verify dataset files exist

Datasets live under `dataset/bc5cdr/` and `dataset/pubmed/`. The configs
expect five files; the cell below lists their sizes.

In [ ]:
from pathlib import Path
for rel in [
    'dataset/bc5cdr/bc5cdr_train.jsonl',
    'dataset/bc5cdr/bc5cdr_validation.jsonl',
    'dataset/bc5cdr/bc5cdr_test.jsonl',
    'dataset/pubmed/pubmed_scrapping.jsonl',
    'dataset/pubmed/pubmed_test.jsonl',
]:
    p = Path(rel)
    print(f"{rel}: {'OK' if p.exists() else 'MISSING'} ({p.stat().st_size if p.exists() else 0} bytes)")

## 4. Pick seeds

Default is 3 seeds for paper-grade mean ± std reporting. Override the
`SEEDS` variable here to change the seed list once for every cell below.

In [ ]:
SEEDS = "42,1337,2024"
print('Using seeds:', SEEDS)

## 5. Config 1 — BERT-base + BC5CDR (lower-bound baseline)

In [ ]:
!bash scripts/run_config_1.sh "$SEEDS"

## 6. Config 2 — BioBERT + BC5CDR

In [ ]:
!bash scripts/run_config_2.sh "$SEEDS"

## 7. Config 3 — PubMedBERT + BC5CDR

In [ ]:
!bash scripts/run_config_3.sh "$SEEDS"

## 8. Config 4 — Sequential (BC5CDR → silver)

Trains 9-tag head; phase 1 sees only Chem/Dis, phase 2 lifts Virus/Gene
from the silver corpus. Forgetting score on Chem/Dis is logged automatically.

In [ ]:
!bash scripts/run_config_4.sh "$SEEDS"

## 9. Config 5 — Joint training (uniform sample weights)

In [ ]:
!bash scripts/run_config_5.sh "$SEEDS"

## 10. Config 6 — Joint training (noise-aware, silver=0.3)

In [ ]:
!bash scripts/run_config_6.sh "$SEEDS"

## 11. View aggregated results

Each `run_config_X.sh` automatically invokes the aggregation script when
all seeds are done. The cell below collates every config's markdown table
into one display for quick comparison.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
for results_dir in sorted(Path('outputs/results').glob('config_*')):
    md_file = results_dir / 'aggregated_results.md'
    if md_file.exists():
        display(Markdown(md_file.read_text(encoding='utf-8')))
    else:
        print(f"[no aggregated_results.md yet] {results_dir.name}")

## 12. (Optional) Inference smoke test

After Config 4 finishes you can run inference on a few PubMed sentences
using the standalone CLI. Change the checkpoint path to whichever you
trained.

In [ ]:
!python predict.py \
    --checkpoint outputs/checkpoints/config_4_sequential/seed_42/phase2/best_model \
    --input dataset/pubmed/pubmed_test.jsonl \
    --output outputs/predictions/config_4_pubmed_test.jsonl